# TradeSight Export Analytics: Repeatable Data Pipeline & Baseline Metrics
**Cynaris Solutions Learning Path | Capstone Project (Data Analyst Intern)**

This Jupyter Notebook demonstrates the **end-to-end repeatable data pipeline** and **baseline metric calculations** for TradeSight Global Export Management (Chennai, India).

### Pipeline Objectives:
1. **Automated ETL:** Extract from raw B2B CSV data packs, clean, enrich with 38 trade features, and validate quality.
2. **Star Schema Data Warehouse:** Ingest into relational SQLite (`tradesight.db`) with production dimension and fact tables.
3. **Baseline Metric Calculations:** Compute C-Suite KPIs, 12-country currency impact, client profitability rankings, and compliance exception distributions.
4. **Repeatable Execution:** Idempotent, testable execution via `run_pipeline.py` and SQL views.

In [1]:
import sys
import os
import sqlite3
import json

# Add workspace to path and execute master pipeline
sys.path.insert(0, os.path.abspath('.'))
import run_pipeline

print("Triggering repeatable pipeline execution...")
run_pipeline.main()

Triggering repeatable pipeline execution...
TRADESIGHT EXPORT ANALYTICS - REPEATABLE DATA PIPELINE
Cynaris Solutions Learning Path | Capstone Project
  [EXTRACT] Reading raw CSV data packs...
    - Customers: 50 records (12 columns)
    - Products:  25 records (9 columns)
    - Orders:    1250 records (31 columns)
  [TRANSFORM] Executing data cleaning and feature engineering...
    - Cleaned orders enriched: 1250 records (38 features)
    - Cleaned datasets saved to: ./data/processed
  [VALIDATE] Running automated data quality test suite...
    - Test Results: 9/9 passed. Status: SUCCESS (All Passed)
  [LOAD] Connecting to database: ./tradesight.db...
    - Star Schema DDL and Analytical Views initialized.
    - Loaded dim_countries: 12 records
    - Loaded dim_customers: 50 records
    - Loaded dim_products:  25 records
    - Loaded fct_export_orders: 1250 records
  [METRICS] Calculating and persisting baseline trade metrics...
    - Baseline metrics persisted to table: agg_baseline_m

## Phase 2: Automated Data Quality Validation
Before loading into the Star Schema, `pipeline/validate.py` executes 9 automated integrity tests:
* **Primary Key Uniqueness:** Verified across `customer_id`, `product_id`, and `order_id`.
* **Referential Integrity:** Confirmed 0 orphaned foreign keys.
* **Chronological Consistency:** Confirmed `dispatch_date >= order_date` and `actual_delivery_date >= dispatch_date`.
* **Domain Whitelists:** Validated 100% adherence to 12 supported countries and 10 settlement currencies.

In [2]:
with open('reports/data_validation_report.json', 'r') as f:
    val_report = json.load(f)

print(f"Validation Status: {'PASSED' if val_report['passed'] else 'FAILED'}")
print(f"Tests Run: {val_report['tests_run']} | Passed: {val_report['tests_passed']}")
print("-" * 60)
for test in val_report['details']:
    print(f"[{test['status']}] {test['test_name']:<35} : {test['message']}")

Validation Status: PASSED
Tests Run: 9 | Passed: 9
------------------------------------------------------------
[PASS] Customer PK Uniqueness              : 50 records, 50 unique IDs
[PASS] Product PK Uniqueness               : 25 records, 25 unique IDs
[PASS] Order PK Uniqueness                 : 1250 records, 1250 unique IDs
[PASS] Order -> Customer FK Integrity      : 0 orphaned customer keys found
[PASS] Order -> Product FK Integrity       : 0 orphaned product keys found
[PASS] Chronological Sequence Integrity    : 0 chronological date discrepancies
[PASS] Positive Values & Exchange Rates    : 0 non-positive values encountered
[PASS] 12 Destination Country Whitelist    : Unsupported countries: []
[PASS] Settlement Currency Whitelist       : Unsupported currencies: []


## Phase 3: Computed C-Suite Baseline Metrics
The metrics engine (`pipeline/metrics.py`) queries the analytical view `vw_c_suite_kpis` and persists benchmark KPIs into `agg_baseline_metrics`.

In [3]:
conn = sqlite3.connect('tradesight.db')
c = conn.cursor()
c.execute('SELECT * FROM vw_c_suite_kpis')
row = c.fetchone()
cols = [d[0] for d in c.description]
for col, val in zip(cols, row):
    if isinstance(val, float):
        print(f"{col:<32}: {val:>15,.2f}")
    else:
        print(f"{col:<32}: {str(val):>15}")
conn.close()

=== C-SUITE EXECUTIVE BASELINE METRICS ===
  total_orders                  : 1250
  total_revenue_booked_inr      : 1725370544.36
  total_revenue_realized_inr    : 1724140493.69
  net_fx_impact_inr             : -1230050.67
  total_gross_profit_inr        : 494730887.74
  gross_profit_margin_pct       : 28.69
  on_time_delivery_rate_pct     : 33.28
  avg_transit_lead_time_days    : 22.5
  compliance_exception_rate_pct : 15.84
  total_active_clients          : 50
  total_destination_countries   : 12


## Phase 4: 12-Country Export Performance & Foreign Exchange (FX) Impact
Measures realized revenue in INR, gross margins, and net foreign exchange gain/loss caused by exchange rate movements between booking and settlement across 10 global currencies.

In [4]:
conn = sqlite3.connect('tradesight.db')
c = conn.cursor()
c.execute('SELECT destination_country, country_code, region, settlement_currency, total_orders, realized_revenue_inr, total_profit_inr, gross_margin_pct, net_fx_impact_inr FROM vw_country_fx_summary')
rows = c.fetchall()
print(f"{'Country':<18} | {'Code':<4} | {'Cur':<4} | {'Orders':<6} | {'Revenue (INR)':<15} | {'Profit (INR)':<13} | {'Margin%':<7} | {'Net FX (INR)':<13}")
print('-' * 95)
for r in rows:
    print(f"{r[0]:<18} | {r[1]:<4} | {r[3]:<4} | {r[4]:<6} | ₹{r[5]:>13,.2f} | ₹{r[6]:>11,.2f} | {r[7]:>6.2f}% | ₹{r[8]:>11,.2f}")
conn.close()

Country            | Code | Cur  | Orders | Revenue (INR)   | Profit (INR)  | Margin% | Net FX (INR) 
-----------------------------------------------------------------------------------------------
Japan              | JPN  | JPY  | 109    | ₹189,225,440.12 | ₹52,801,619.80 |  27.90% | ₹  69,413.88
Singapore          | SGP  | SGD  | 116    | ₹169,836,440.14 | ₹49,387,218.30 |  29.08% | ₹-519,560.36
United States      | USA  | USD  | 119    | ₹166,832,909.22 | ₹46,895,802.45 |  28.11% | ₹   8,380.30
Australia          | AUS  | AUD  | 100    | ₹162,336,174.68 | ₹46,530,578.81 |  28.66% | ₹ 438,907.87
United Kingdom     | GBR  | GBP  | 103    | ₹160,104,571.13 | ₹46,004,994.93 |  28.73% | ₹ 124,811.02
South Africa       | ZAF  | ZAR  | 110    | ₹158,494,614.76 | ₹46,550,177.66 |  29.37% | ₹ 344,158.72
Netherlands        | NLD  | EUR  | 106    | ₹156,066,224.19 | ₹47,347,065.04 |  30.34% | ₹ 173,968.84
Germany            | DEU  | EUR  | 85     | ₹130,680,357.18 | ₹35,708,076.14 |  27.32% |

## Phase 5: Client Profitability Ranking (Top 10 Accounts)
Ranks all 50 active B2B clients by net gross margin and realized profit contribution to prioritize commercial focus.

In [5]:
conn = sqlite3.connect('tradesight.db')
c = conn.cursor()
c.execute('SELECT * FROM vw_client_profitability_ranked LIMIT 10')
rows = c.fetchall()
print(f"{'Rank':<4} | {'Company Name':<30} | {'Country':<15} | {'Tier':<12} | {'Orders':<6} | {'Profit (INR)':<13} | {'Margin%':<7}")
print('-' * 100)
for i, r in enumerate(rows, 1):
    tier = r[4].split(' - ')[1] if ' - ' in r[4] else r[4]
    print(f"#{i:<3} | {r[1]:<30} | {r[2]:<15} | {tier:<12} | {r[6]:<6} | ₹{r[8]:>11,.2f} | {r[9]:>6.2f}%")
conn.close()

Rank | Company Name                   | Country         | Tier         | Orders | Profit (INR)  | Margin%
----------------------------------------------------------------------------------------------------
#1   | Aussie Bulk Distributors       | Japan           | Growth       | 41     | ₹25,738,223.58 |  28.40%
#2   | Sharjah Trading House          | South Africa    | Enterprise   | 36     | ₹23,313,473.69 |  29.05%
#3   | Lyon Precision Tools           | Netherlands     | Growth       | 37     | ₹22,770,736.59 |  29.07%
#4   | Gold Coast Merchants           | Singapore       | Strategic    | 37     | ₹21,000,846.75 |  29.35%
#5   | Emirates Trade Network         | United Kingdom  | Growth       | 23     | ₹16,568,541.68 |  29.46%
#6   | East Asia Freight Logistics    | United States   | Enterprise   | 25     | ₹15,914,876.09 |  27.51%
#7   | Pacific Rim Trade Corp         | Singapore       | Enterprise   | 33     | ₹15,767,912.70 |  28.55%
#8   | Pretoria Industrial Supplies   | Fran

## Phase 6: Export Compliance Exception Risk Register
Analyzes the 198 exception-flagged shipments across 6 trigger categories to prepare for automated weekly alert scheduling.

In [6]:
conn = sqlite3.connect('tradesight.db')
c = conn.cursor()
c.execute('SELECT * FROM vw_compliance_risk_register')
rows = c.fetchall()
print(f"{'Compliance Flag Type':<45} | {'Severity':<10} | {'Count':<6} | {'Share%':<7} | {'Avg Delay':<10}")
print('-' * 90)
for r in rows:
    print(f"{r[0]:<45} | {r[1]:<10} | {r[2]:<6} | {r[3]:>5.2f}% | {r[4]:>5.1f} days")
conn.close()

Compliance Flag Type                          | Severity   | Count  | Share%  | Avg Delay 
------------------------------------------------------------------------------------------
None                                          | None       | 1052   | 84.16% |   0.0 days
Customs Inspection Hold                       | Critical   | 40     |  3.20% |   8.3 days
HS Code Discrepancy                           | Moderate   | 37     |  2.96% |   7.4 days
Fumigation / Phytosanitary Certificate Delay  | Moderate   | 34     |  2.72% |   7.3 days
Documentation Incomplete                      | Moderate   | 32     |  2.56% |   8.1 days
Tariff Rate Query                             | Low        | 30     |  2.40% |   6.8 days
Country Sanctions Screening                   | Critical   | 25     |  2.00% |   7.8 days
